In [ ]:
"""
Model Training
======================================

Trains visitation prediction model on training data.
Tests multiple models × transforms, selects best via cross-validation.
"""

import sys
import warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

# Resolve directory of this file (works in .py; falls back to CWD in notebooks)
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

# Output and input paths relative to this notebook
OUTDIR = (HERE / "../outputs/model_outputs_ca").resolve()
OUTDIR.mkdir(parents=True, exist_ok=True)

# Input: clean training data (from prepare_ca_training_data.py)
TRAINING_DATA = (HERE / "../data/CA_training_data_clean.csv").resolve()

# Output
LOG_FILE = OUTDIR / f"training_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

TARGET = "monthly_visits"
USE_ONLY_RAW = False  # Set to True to skip transforms

# Boruta parameters
BORUTA_ALPHA = 0.02
BORUTA_PERC = 98
BORUTA_MAX_ITER = 300
INCLUDE_TENTATIVE = False
MIN_FEATURES_KEEP = 25

# ============================================================
# IMPORTS
# ============================================================

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import PowerTransformer
from sklearn.base import clone
import joblib
from boruta import BorutaPy
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


# ============================================================
# LOGGING
# ============================================================

def log(message, also_print=True):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    full = f"[{ts}] {message}"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(full + "\n")
    if also_print:
        print(full)

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def coerce_numeric(df):
    """Convert all columns to numeric, fill NaN with median"""
    d = df.copy()
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]):
            d[c] = pd.to_numeric(d[c], errors="coerce")
    for c in d.columns:
        if pd.api.types.is_integer_dtype(d[c]) or pd.api.types.is_bool_dtype(d[c]):
            d[c] = d[c].astype("float64")
    
    num = d.select_dtypes(include=[np.number]).columns
    for c in num:
        if d[c].isna().any():
            med = d[c].median()
            if pd.isna(med):
                med = 0.0
            d[c] = d[c].fillna(med)
    
    d = d.replace([np.inf, -np.inf, np.nan], 0.0)
    return d

# ============================================================
# TRANSFORMS
# ============================================================

def make_transforms(y_raw):
    """Create target transforms"""
    y_raw = np.asarray(y_raw, dtype=float)
    y_raw = np.where(np.isfinite(y_raw), y_raw, 0.0)
    y_raw = np.clip(y_raw, 0.0, None)
    
    # Identity
    def f_id(x): return x
    def inv_id(x): return x
    
    # Square root
    def f_sqrt(x): return np.sqrt(np.clip(x, 0, None))
    def inv_sqrt(x): return np.clip(x, 0, None)**2
    
    # Log1p
    def f_log1p(x): return np.log1p(np.clip(x, 0, None))
    def inv_log1p(x): return np.expm1(x)
    
    # Yeo-Johnson
    pt = PowerTransformer(method="yeo-johnson", standardize=False)
    try:
        pt.fit(y_raw.reshape(-1,1))
        def f_yj(x):   return pt.transform(np.asarray(np.clip(x, 0, None)).reshape(-1,1)).ravel()
        def inv_yj(x): return np.clip(pt.inverse_transform(np.asarray(x).reshape(-1,1)).ravel(), 0, None)
    except:
        def f_yj(x): return x
        def inv_yj(x): return x
    
    # Tweedie
    offset = np.median(y_raw[y_raw > 0]) * 0.01 if np.any(y_raw > 0) else 1.0
    def f_tweedie(x): return np.log(np.clip(x, 0, None) + offset)
    def inv_tweedie(x): return np.exp(x) - offset
    
    return {
        "identity":    {"y": f_id(y_raw),      "inverse": inv_id,      "name": "Identity (raw)"},
        "sqrt":        {"y": f_sqrt(y_raw),    "inverse": inv_sqrt,    "name": "Square root"},
        "log1p":       {"y": f_log1p(y_raw),   "inverse": inv_log1p,   "name": "Log1p"},
        "yeo_johnson": {"y": f_yj(y_raw),      "inverse": inv_yj,      "name": "Yeo-Johnson"},
        "tweedie":     {"y": f_tweedie(y_raw), "inverse": inv_tweedie, "name": "Tweedie-like"},
    }

# ============================================================
# MODEL ZOO
# ============================================================
def get_model_zoo(random_state=42):

    zoo = {

        "RF_Standard": RandomForestRegressor(
            n_estimators=700,
            max_depth=20,
            min_samples_split=4,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1
        ),

        "RF_Deep": RandomForestRegressor(
            n_estimators=900,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=random_state,
            n_jobs=-1
        ),

        "GBRT": GradientBoostingRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=4,
            min_samples_split=8,
            min_samples_leaf=4,
            subsample=0.8,
            random_state=random_state
        ),

        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=700,
            max_depth=25,
            min_samples_split=4,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1
        ),

        "HistGB": HistGradientBoostingRegressor(
            max_iter=500,
            learning_rate=0.05,
            max_depth=8,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=random_state
        ),

        "XGBoost": XGBRegressor(
            n_estimators=700,
            learning_rate=0.05,
            max_depth=6,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=0.1,
            random_state=random_state,
            n_jobs=-1,
            verbosity=0
        ),

        "LightGBM": LGBMRegressor(
            n_estimators=700,
            learning_rate=0.05,
            max_depth=8,
            num_leaves=31,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1
        ),

        "CatBoost": CatBoostRegressor(
            iterations=700,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=3,
            subsample=0.8,
            random_state=random_state,
            verbose=False,
            thread_count=-1
        ),
    }

    return zoo


# ============================================================
# BORUTA FEATURE SELECTION
# ============================================================

def boruta_select(X, y_t, random_state=42):
    if not BORUTA_AVAILABLE:
        return list(X.columns), None
    
    # Pre-filter zero variance
    variances = X.var()
    variable_cols = variances[variances > 0.001].index.tolist()
    
    # Always keep mobility
    for k in {"log_pud","log_aud","log_eud","log_rud","rud_x_available"}:
        if k in X.columns and k not in variable_cols:
            variable_cols.append(k)
    
    X_var = X[variable_cols]
    n_removed = X.shape[1] - len(variable_cols)
    if n_removed > 0:
        log(f"      Pre-filtered {n_removed} near-zero variance features")
    
    # Run Boruta
    base_rf = RandomForestRegressor(
        n_estimators=500, max_depth=25, min_samples_split=4, min_samples_leaf=2,
        random_state=random_state, n_jobs=-1
    )
    sel = BorutaPy(
        estimator=base_rf, n_estimators="auto", max_iter=BORUTA_MAX_ITER,
        alpha=BORUTA_ALPHA, perc=BORUTA_PERC, two_step=True,
        random_state=random_state, verbose=0
    )
    sel.fit(X_var.values.astype(float), np.asarray(y_t, dtype=float))
    
    confirmed = sel.support_
    tentative = getattr(sel, "support_weak_", np.zeros_like(confirmed, dtype=bool))
    mask = confirmed if not INCLUDE_TENTATIVE else (confirmed | tentative)
    feats = [X_var.columns[i] for i, keep in enumerate(mask) if keep]
    
    # Ensure minimum features
    if len(feats) < max(MIN_FEATURES_KEEP, int(0.25 * X.shape[1])):
        tmp = RandomForestRegressor(n_estimators=600, max_depth=25, random_state=random_state, n_jobs=-1)
        tmp.fit(X_var, y_t)
        imp = pd.Series(tmp.feature_importances_, index=X_var.columns).sort_values(ascending=False)
        need = max(MIN_FEATURES_KEEP, int(0.25 * X.shape[1])) - len(feats)
        pad = [f for f in imp.index if f not in feats][:max(0, need)]
        feats = feats + pad
    
    return feats, sel

# ============================================================
# MAIN
# ============================================================

def main():
    log("="*70)
    log("CALIFORNIA: MODEL TRAINING")
    log("="*70)
    
    # Load clean training data
    log(f"\nLoading clean training data: {TRAINING_DATA}")
    df = pd.read_csv(TRAINING_DATA)
    log(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # Check target
    if TARGET not in df.columns:
        log(f"ERROR: Target '{TARGET}' not found")
        return False
    
    # Build X/y
    y_raw = pd.to_numeric(df[TARGET], errors="coerce").fillna(0.0).values
    
    model_features = filter_features(list(df.columns))
    drop_like = {TARGET, f"sqrt_{TARGET}", f"log_{TARGET}", f"yj_{TARGET}"}
    model_features = [c for c in model_features if c not in drop_like]
    
    X_all = coerce_numeric(df[model_features].copy())
    
    # Setup CV
    groups = df["siteid"].astype(str).values if "siteid" in df.columns else np.array(["ALL"] * len(df))
    n_groups = pd.Series(groups).nunique()
    n_splits = 3 if n_groups >= 3 else max(2, n_groups)
    gkf = GroupKFold(n_splits=n_splits)
    splits = list(gkf.split(X_all, groups=groups))
    
    log(f"\nData summary:")
    log(f"  n={len(X_all):,}, p={X_all.shape[1]}, groups={n_groups}, folds={n_splits}")
    log(f"  Target range: [{y_raw.min():.1f}, {y_raw.max():.1f}]")
    log(f"  Boruta: {'ON' if BORUTA_AVAILABLE else 'OFF'}")
    
    # Test transforms × models
    transforms = {"identity": {"y": y_raw, "inverse": (lambda x: x), "name": "Identity (raw)"}} if USE_ONLY_RAW else make_transforms(y_raw)
    zoo = get_model_zoo(42)
    by_transform = []
    
    log(f"\n{'='*20} TESTING {len(zoo)} MODELS × {len(transforms)} TRANSFORMS {'='*20}")
    
    for tname, T in transforms.items():
        log(f"\n{tname.upper()} ({T['name']})")
        y_t = T["y"]
        
        # Feature selection
        if BORUTA_AVAILABLE:
            log("   Boruta feature selection...")
            feats, _ = boruta_select(X_all, y_t, random_state=42)
            X_t = X_all[feats].copy()
            log(f"   Using {X_t.shape[1]} features")
        else:
            X_t = X_all.copy()
            feats = list(X_t.columns)
            log(f"   Using all {X_t.shape[1]} features")
        
        # Test each model
        results_cfg = {}
        for name, est in zoo.items():
            y_pred_all, y_true_all, fold_scores = [], [], []
            
            for k, (tr, te) in enumerate(splits, 1):
                est_k = clone(est)
                est_k.fit(X_t.iloc[tr], y_t[tr])
                yhat = est_k.predict(X_t.iloc[te])
                
                fold_scores.append({
                    "fold": k,
                    "r2": r2_score(y_t[te], yhat),
                    "rmse": rmse(y_t[te], yhat),
                    "mae": mean_absolute_error(y_t[te], yhat)
                })
                y_pred_all.extend(yhat)
                y_true_all.extend(y_t[te])
            
            cv_mean = float(np.mean([s["r2"] for s in fold_scores]))
            cv_std = float(np.std([s["r2"] for s in fold_scores]))
            overall_r2 = r2_score(y_true_all, y_pred_all)
            overall_rmse = rmse(y_true_all, y_pred_all)
            
            results_cfg[name] = dict(
                cv_mean=cv_mean, cv_std=cv_std,
                overall_r2=overall_r2, overall_rmse=overall_rmse,
                folds=fold_scores
            )
            log(f"   {name:<12} | CV R²= {cv_mean:>6.3f}±{cv_std:>5.3f} | OOF R²={overall_r2:>6.3f}")
        
        best_key = max(results_cfg.keys(), key=lambda k: results_cfg[k]["overall_r2"])
        best = results_cfg[best_key]
        log(f"   Best: {best_key} (OOF R²={best['overall_r2']:.3f})")
        
        by_transform.append(dict(
            transform=tname, name=T["name"], feats=feats, X_t=X_t,
            best_model_name=best_key, cv_best=best, inverse=T["inverse"]
        ))
    
    # Summary
    summary = pd.DataFrame([{
        "transform": p["transform"], "name": p["name"], "best_model": p["best_model_name"],
        "cv_mean": p["cv_best"]["cv_mean"], "cv_std": p["cv_best"]["cv_std"],
        "overall_r2": p["cv_best"]["overall_r2"], "overall_rmse": p["cv_best"]["overall_rmse"],
        "n_features": len(p["feats"])
    } for p in by_transform]).sort_values("overall_r2", ascending=False).reset_index(drop=True)
    
    log("\n" + "="*70)
    log("RESULTS (ranked by OOF R²)")
    log("="*70)
    log(summary[["transform","name","best_model","overall_r2","cv_mean","n_features"]].to_string(index=False))
    
    best_row = summary.iloc[0]
    Winner = next(p for p in by_transform if p["transform"] == best_row["transform"])
    log(f"\nWINNER: {Winner['name']} + {Winner['best_model_name']}")
    log(f"   OOF R²={Winner['cv_best']['overall_r2']:.3f}")
    
    # Train final model
    Xw = coerce_numeric(df[Winner["feats"].copy()])
    y_win = transforms[Winner["transform"]]["y"]
    
    final_template = get_model_zoo(42)[Winner["best_model_name"]]
    
    # Generate CV predictions
    log("\nGenerating CV predictions...")
    cv = GroupKFold(n_splits=n_splits)
    all_preds_t, all_actuals_t, all_sites, all_fold, all_indices = [], [], [], [], []
    
    for k, (tr, te) in enumerate(cv.split(Xw, y_win, groups=groups), 1):
        est_k = clone(final_template)
        est_k.fit(Xw.iloc[tr], y_win[tr])
        yp = est_k.predict(Xw.iloc[te])
        
        all_preds_t.extend(yp)
        all_actuals_t.extend(y_win[te])
        all_sites.extend(df["siteid"].iloc[te].astype(str) if "siteid" in df.columns else ["ALL"]*len(te))
        all_fold.extend([f"Fold {k}"]*len(te))
        all_indices.extend(te)
    
    cv_overall_r2 = r2_score(all_actuals_t, all_preds_t)
    
    # Save CV predictions
    cv_df = pd.DataFrame({
        "siteid": all_sites,
        "fold": all_fold,
        "y_true_t": all_actuals_t,
        "y_pred_t": all_preds_t
    })
    if "year" in df.columns and "month" in df.columns:
        cv_df["year"] = df["year"].iloc[all_indices].values
        cv_df["month"] = df["month"].iloc[all_indices].values
    cv_df.to_csv(OUTDIR / "cv_predictions.csv", index=False)
    
    # Train final model on all data
    log("\nTraining final model on all data...")
    final_model = clone(final_template)
    final_model.fit(Xw, y_win)
    
    # Train all models for feature importance comparison
    log("\nTraining all models for feature importance...")
    all_models_features = {}
    zoo_all = get_model_zoo(42)
    
    for model_name, model_template in zoo_all.items():
        log(f"   {model_name}...")
        model_fitted = clone(model_template)
        model_fitted.fit(Xw, y_win)
        
        feat_imp = None
        if hasattr(model_fitted, "feature_importances_"):
            feat_imp = model_fitted.feature_importances_
        else:
            try:
                from sklearn.inspection import permutation_importance
                perm = permutation_importance(model_fitted, Xw, y_win, n_repeats=10, random_state=42, n_jobs=-1)
                feat_imp = perm.importances_mean
            except:
                pass
        
        if feat_imp is not None:
            feat_imp = feat_imp / (feat_imp.sum() + 1e-10)
            all_models_features[model_name] = {"importance": feat_imp, "features": Winner["feats"]}
            pd.DataFrame({"feature": Winner["feats"], "importance": feat_imp}) \
              .sort_values("importance", ascending=False) \
              .to_csv(OUTDIR / f"feature_importance_{model_name}.csv", index=False)
    
    # Save model
    joblib.dump({
        "model": final_model,
        "features": Winner["feats"],
        "X_train": Xw,
        "y_train": y_win,
        "best_model_name": Winner["best_model_name"],
        "transform": Winner["transform"],
        "cv_performance": {
            "overall_r2": cv_overall_r2,
            "cv_mean": Winner["cv_best"]["cv_mean"],
            "cv_std": Winner["cv_best"]["cv_std"]
        },
        "config": {
            "n_features": len(Winner["feats"]),
            "n_obs": len(Xw),
            "n_sites": int(n_groups)
        },
        "all_models_features": all_models_features
    }, OUTDIR / "final_model.joblib")
    
    log(f"\n{'='*70}")
    log(f"COMPLETE! OOF R²={cv_overall_r2:.3f}")
    log(f"Model saved: {OUTDIR / 'final_model.joblib'}")
    log(f"{'='*70}")
    
    return True

if __name__ == "__main__":
    main()
    print("✓ Finished successfully")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
"